# 12 — Refactoring: From Notebook Code to a Module

By this point you have written—or seen—the Haversine formula in at least **four** different notebooks:

| Module | Notebook | Function name |
|--------|----------|---------------|
| 06 | `01-Haversine_Distance.ipynb` | `haversine_km(p1, p2)` |
| 07 | `01-Compute_Bearing.ipynb` | *(imported inline)* |
| 08 | `01-Constant_Velocity_Intercept.ipynb` | `haversine_km(p1, p2)` |
| 10 | `01_Buffering_Points.ipynb` | `haversine_km(p1, p2)` |

That repetition is the subject of this lesson.  
We are going to look at **why it is a problem**, and **what to do about it**.

---
## 1 — The Copy-Paste Problem

Here are two real copies of `haversine_km` from different notebooks in this course, placed side by side.

In [ ]:
# Version A — from Module 06
import math

def haversine_km_A(p1, p2):
    """p1, p2 are [lon, lat] pairs."""
    lon1, lat1 = p1
    lon2, lat2 = p2
    R = 6371
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2 * R * math.asin(math.sqrt(a))


# Version B — from Module 08
def haversine_km_B(p1, p2):
    """p1, p2 are [lon, lat] pairs."""
    lon1, lat1 = p1
    lon2, lat2 = p2
    R = 6371.0088          # <-- slightly different constant
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))   # <-- atan2 instead of asin
    return R * c


# Same inputs.  Do they agree?
shooter = [-97.3, 33.9]   # [lon, lat]
target  = [-94.1, 36.2]

print(f"Version A: {haversine_km_A(shooter, target):.4f} km")
print(f"Version B: {haversine_km_B(shooter, target):.4f} km")
print(f"Difference: {abs(haversine_km_A(shooter, target) - haversine_km_B(shooter, target)):.4f} km")

The two versions return slightly different answers.  
Which one is correct?  Why are they different?  If you find a bug, how many notebooks do you have to fix?

> **The rule of three:** the first time you write something, write it.  The second time, note the duplication.  The third time, **refactor**.

---
## 2 — What Is a Module?

A **module** is just a `.py` file.  When Python runs `import geo`, it executes that file and makes everything defined in it available under the name `geo`.

```
project/
  my_notebook.ipynb
  geo.py            ← the module
```

Anything you put in `geo.py` can be imported:

```python
from geo import haversine_km, compute_bearing
```

That's it.  No magic.  Let's build one.

In [ ]:
# We'll write a small geo.py right here using Python's file I/O
# so you can see exactly what ends up on disk.

geo_src = '''
"""geo.py — a minimal geodetic utility module."""
from math import radians, degrees, sin, cos, asin, atan2, sqrt

EARTH_RADIUS_KM = 6371.0088  # mean radius — write this constant ONCE


def haversine_km(lat1, lon1, lat2, lon2):
    """
    Great-circle distance between two points.
    Inputs: degrees.  Output: kilometers.
    """
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi   = radians(lat2 - lat1)
    dlambda = radians(lon2 - lon1)
    a = sin(dphi/2)**2 + cos(phi1)*cos(phi2)*sin(dlambda/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return EARTH_RADIUS_KM * c
'''

with open("geo.py", "w") as f:
    f.write(geo_src)

print("geo.py written.")

In [ ]:
# Now import from it — just like any other library
from geo import haversine_km, EARTH_RADIUS_KM

lat1, lon1 = 33.9, -97.3
lat2, lon2 = 36.2, -94.1

print(f"Distance: {haversine_km(lat1, lon1, lat2, lon2):.4f} km")
print(f"Earth radius used: {EARTH_RADIUS_KM} km")

Notice the signature changed: `haversine_km(lat1, lon1, lat2, lon2)` — four separate floats, not two lists.  
That brings us to the next problem.

---
## 3 — The Coordinate Order Problem

Throughout the notebooks, points have been represented in two different ways:

```python
# GeoJSON convention (used in Modules 01–04, 10)
point = [lon, lat]        # [-97.3, 33.9]

# Math / human convention (used in some formulas)
point = (lat, lon)        # (33.9, -97.3)
```

Both exist in the wild.  The danger is silently passing one where the other is expected:

In [ ]:
# Correct call
d_correct = haversine_km(33.9, -97.3, 36.2, -94.1)

# Accidentally swapped — lon fed where lat is expected
d_swapped = haversine_km(-97.3, 33.9, -94.1, 36.2)

print(f"Correct:  {d_correct:.1f} km")
print(f"Swapped:  {d_swapped:.1f} km")   # wrong — but Python won't complain

Python cannot catch that bug.  The numbers are valid floats; the answer is just wrong.

One solution is a **typed point object** that makes the field names explicit.

---
## 4 — Introducing a Dataclass for Points

A `dataclass` is a lightweight class whose job is to hold named data fields.  
No math, no logic — just a named container.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)   # frozen=True makes it immutable, like a tuple
class LatLon:
    lat: float
    lon: float


# Create a point — field names make the order obvious
shooter = LatLon(lat=33.9, lon=-97.3)
target  = LatLon(lat=36.2, lon=-94.1)

print(shooter)
print(f"lat={shooter.lat}, lon={shooter.lon}")

In [ ]:
# Update haversine_km to accept LatLon objects
def haversine_km_v2(p1: LatLon, p2: LatLon) -> float:
    from math import radians, sin, cos, atan2, sqrt
    EARTH_RADIUS_KM = 6371.0088

    phi1, phi2 = radians(p1.lat), radians(p2.lat)
    dphi    = radians(p2.lat - p1.lat)
    dlambda = radians(p2.lon - p1.lon)
    a = sin(dphi/2)**2 + cos(phi1)*cos(phi2)*sin(dlambda/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return EARTH_RADIUS_KM * c


print(f"{haversine_km_v2(shooter, target):.4f} km")

Now you cannot accidentally pass a `[lon, lat]` list — Python will raise `AttributeError: 'list' object has no attribute 'lat'` immediately.  That is the error you *want* to catch early.

> **Key idea:** a `dataclass` is not a class because the problem is complex.  It is a class because **named fields prevent a whole category of silent bugs**.

---
## 5 — When to Use a Class vs. Plain Functions

Not every refactoring needs a class.  A good rule of thumb:

| Situation | Prefer |
|-----------|--------|
| Stateless calculation (distance, bearing) | **Module function** |
| Named data with no behavior | **Dataclass** |
| Problem with setup parameters and multiple solve methods | **Class** |
| UI state (map layers, widget callbacks) | **Class or enclosure** |

Haversine is stateless — it takes inputs and returns an output every time.  A plain function is correct.

The intercept problem from Module 08 is different.  You set it up once (shooter position, target speed, heading) and then might call `find_intercept_time`, `compute_intercept`, and `simulate_pursuit` all on the same configured problem.  That is a natural fit for a class.

In [ ]:
# Compare: stateless function (fine as a module function)
def compute_bearing(p1: LatLon, p2: LatLon) -> float:
    """Compass bearing from p1 to p2 in degrees [0, 360)."""
    from math import radians, degrees, sin, cos, atan2
    phi1 = radians(p1.lat)
    phi2 = radians(p2.lat)
    dlambda = radians(p2.lon - p1.lon)
    y = sin(dlambda) * cos(phi2)
    x = cos(phi1) * sin(phi2) - sin(phi1) * cos(phi2) * cos(dlambda)
    return degrees(atan2(y, x)) % 360


# Stateful problem — a class keeps the parameters together
class InterceptProblem:
    def __init__(self, shooter: LatLon, target: LatLon,
                 target_heading_deg: float,
                 target_speed_kmh: float,
                 shooter_speed_kmh: float):
        self.shooter = shooter
        self.target  = target
        self.target_heading = target_heading_deg
        self.target_speed   = target_speed_kmh
        self.shooter_speed  = shooter_speed_kmh

    def can_intercept(self) -> bool:
        """A pursuer that is slower than the target can never catch it."""
        return self.shooter_speed > self.target_speed

    # find_intercept_time, compute_intercept, simulate_pursuit
    # would all live here, sharing self.shooter, self.target, etc.


problem = InterceptProblem(
    shooter=LatLon(33.9, -97.3),
    target=LatLon(36.2, -94.1),
    target_heading_deg=210,
    target_speed_kmh=800,
    shooter_speed_kmh=1200,
)
print(f"Can intercept: {problem.can_intercept()}")
print(f"Initial bearing to target: {compute_bearing(problem.shooter, problem.target):.1f}°")

---
## 6 — Separating Geometry from Map UI

One of the easiest wins in refactoring is pulling **pure geometry code** out of **map display code**.

Look at this pattern from Module 04:

In [ ]:
# BEFORE: geometry tangled with UI
# (pseudocode — don't run)

def on_click(**kwargs):
    if kwargs.get("type") != "click":
        return
    lat, lon = kwargs["coordinates"]

    # Geometry logic buried inside a UI callback
    for other_lat, other_lon in stored_points:
        d = haversine_km(lat, lon, other_lat, other_lon)  # geometry
        if d < 50:
            print(f"Within 50 km of stored point!")

    # Map mutation
    stored_points.append((lat, lon))
    m.add_layer(Marker(location=(lat, lon)))

In [ ]:
# AFTER: geometry is a pure function, UI callback stays thin

def points_within_radius(new_point: LatLon, existing: list, radius_km: float) -> list:
    """Pure geometry — no map, no widgets, easy to test."""
    return [
        p for p in existing
        if haversine_km_v2(new_point, p) < radius_km
    ]


# The callback now just calls the geometry function
def on_click_v2(**kwargs):
    if kwargs.get("type") != "click":
        return
    lat, lon = kwargs["coordinates"]
    clicked = LatLon(lat, lon)

    nearby = points_within_radius(clicked, stored_latlon_points, radius_km=50)
    if nearby:
        print(f"Within 50 km of {len(nearby)} stored point(s)!")

    stored_latlon_points.append(clicked)
    # m.add_layer(...)


# Test the geometry without ever opening a map
stored_latlon_points = [LatLon(33.9, -97.3), LatLon(36.2, -94.1)]
test_point = LatLon(34.5, -96.0)
nearby = points_within_radius(test_point, stored_latlon_points, radius_km=200)
print(f"Nearby points: {nearby}")

`points_within_radius` can be called in a test cell without any map running.  
That makes bugs much easier to find.

---
## 7 — What `wdo/geo.py` Actually Contains

In this project the refactored geometry lives at `src/wdo/geo.py`.  
You don't need to write it from scratch — it already exists and you will import it in Module 13.  
But understanding what went into it makes the import meaningful.

Here's the structure at a glance:

```
src/wdo/geo.py
│
├── EARTH_RADIUS_KM = 6371.0088      # one authoritative constant
│
├── LatLon(lat, lon)                  # dataclass — named coordinates
│
├── haversine_km(lat1, lon1, lat2, lon2)       # great-circle distance
├── initial_bearing_deg(lat1, lon1, lat2, lon2) # compass bearing
├── destination_point(lat, lon, bearing_deg, distance_km)  # inverse haversine
│
├── bbox_latlon(points)              # bounding box from a list of LatLon
├── orientation(p, q, r)             # cross-product sign
├── segments_intersect(p1, q1, p2, q2)
├── point_in_polygon(point, polygon)
│
└── point_feature / line_feature / polygon_feature / feature_collection
                                     # GeoJSON builder helpers
```

Every function you have written across Modules 05–11 is represented here — once, tested, with consistent `(lat, lon)` ordering throughout.

---
## 8 — Refactoring Exercise

Below is a copy of `point_in_ring` from Module 11.  It uses `[lon, lat]` lists.

In [ ]:
# Original — from Module 11
def point_in_ring(lon, lat, ring):
    """
    Ray-casting point-in-polygon test.
    ring: list of [lon, lat] pairs.
    """
    inside = False
    n = len(ring)
    for i in range(n):
        xi, yi = ring[i]
        xj, yj = ring[(i + 1) % n]
        crosses = (yi > lat) != (yj > lat)
        if crosses:
            x_intersect = xi + (lat - yi) * (xj - xi) / (yj - yi)
            if lon < x_intersect:
                inside = not inside
    return inside


# Quick smoke test with the original signature
ring = [[-100,35],[-100,40],[-95,40],[-95,35],[-100,35]]
print(point_in_ring(-97.0, 37.5, ring))   # True — inside
print(point_in_ring(-110.0, 37.5, ring))  # False — outside

In [ ]:
# YOUR TURN
# Refactor point_in_ring so that:
#   1. The point is a LatLon object (not separate lon, lat args)
#   2. The ring is a list of LatLon objects (not [lon,lat] lists)
#   3. The function still returns the correct True/False answers above

def point_in_ring_v2(point: LatLon, ring: list) -> bool:
    inside = False
    closed_ring = ring if ring[0] == ring[-1] else ring + [ring[0]]
    for i in range(len(closed_ring) - 1):
        a = closed_ring[i]
        b = closed_ring[i + 1]
        if (a.lat > point.lat) != (b.lat > point.lat):
            lon_cross = a.lon + (point.lat - a.lat) * (b.lon - a.lon) / (b.lat - a.lat)
            if point.lon < lon_cross:
                inside = not inside
    return inside


ring_v2 = [
    LatLon(lat=35, lon=-100),
    LatLon(lat=40, lon=-100),
    LatLon(lat=40, lon=-95),
    LatLon(lat=35, lon=-95),
]
print(point_in_ring_v2(LatLon(37.5, -97.0), ring_v2))   # True
print(point_in_ring_v2(LatLon(37.5, -110.0), ring_v2))  # False

---
## Summary

| Problem | Refactoring move |
|---------|------------------|
| Same function copy-pasted across notebooks | Extract to a `.py` module, `import` everywhere |
| Inconsistent coordinate order | Choose one convention; use named fields (`LatLon`) |
| Magic constant (`6371`) repeated | Define `EARTH_RADIUS_KM` once at module level |
| Geometry buried in UI callbacks | Extract pure functions, keep callbacks thin |
| Related setup + multiple solve methods | Bundle into a class (`InterceptProblem`) |

These moves do not change what the code computes.  They change how easy it is to read, test, and fix.

---
## Check Your Understanding

You have two functions:

```python
def haversine_km(lat1, lon1, lat2, lon2): ...      # A
def nearest_n(base, named_points, n): ...           # B
```

Function A is purely mathematical — same inputs always produce the same output.  
Function B loops over a list of named points and returns the `n` closest ones.

**Question:** Should either of these become a method on a class?  If so, which one and why?  If not, why not?

Write your answer in the cell below.

Function A should stay a plain function because it is pure math with no stored state. Function B can also stay a function if it just receives a list and returns the nearest results, but it becomes a good method when the named-point dataset is something you want to keep attached to an object and reuse across many queries.